In [63]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
import os

In [64]:
load_dotenv(override=True)

True

In [65]:
model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.5,
)

In [66]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str
    rating: int

In [67]:
def create_outline(state):
    title = state['title']

    prompt = f"Create a detailed outline for a blog on the topic: {title}"

    response = model.invoke(prompt)

    state['outline'] = response.text

    return state

In [68]:
def create_blog(state):
    title = state['title']
    outline = state['outline']

    prompt = f"""
    Write a detailed blog on the title - {title}
    using the following outline:

    {outline}
    """

    response = model.invoke(prompt)

    state['content'] = response.text

    return state

In [69]:
def evaluate(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']
    content = state['content']

    prompt = f"""
    Evaluate the blog on the basis of:

    Title: {title}
    Outline: {outline}
    Content: {content}

    Rate the blog out of 10.
    Return ONLY the rating as a number.
    """

    rating = model.invoke(prompt).text

    state['rating'] = int(rating)

    return state

In [70]:
graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate', evaluate)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate')
graph.add_edge('evaluate', END)

workflow = graph.compile()

In [71]:
initial_state = {'title': "Rise of AI in India"}
final_state = workflow.invoke(initial_state)

print(final_state)


{'title': 'Rise of AI in India', 'outline': 'Here is a comprehensive, highly detailed blog outline on the **"Rise of AI in India."** \n\nThis outline is structured to be SEO-friendly, engaging, and easy to write, complete with suggested keywords, section-by-section breakdown, visual ideas, and writing tips.\n\n---\n\n# Blog Title Ideas:\n*   *The Silicon Valley of the East: Inside India’s Massive AI Revolution*\n*   *From IT Hub to AI Powerhouse: The Rise of Artificial Intelligence in India*\n*   *AI for All: How India is Democratizing Artificial Intelligence*\n*   *The Rise of AI in India: Startups, Sovereign Tech, and the Future of Work*\n\n---\n\n### **Target Keywords:**\n*   *Primary:* Rise of AI in India, AI in India, Indian AI startups\n*   *Secondary:* IndiaAI Mission, Indic LLMs, AI in Indian healthcare, tech talent in India, digital revolution India\n\n---\n\n## **Detailed Blog Outline**\n\n### **1. Introduction (Approx. 150–200 words)**\n*   **The Hook:** Start with a compell

In [72]:
print(final_state['outline'])

Here is a comprehensive, highly detailed blog outline on the **"Rise of AI in India."** 

This outline is structured to be SEO-friendly, engaging, and easy to write, complete with suggested keywords, section-by-section breakdown, visual ideas, and writing tips.

---

# Blog Title Ideas:
*   *The Silicon Valley of the East: Inside India’s Massive AI Revolution*
*   *From IT Hub to AI Powerhouse: The Rise of Artificial Intelligence in India*
*   *AI for All: How India is Democratizing Artificial Intelligence*
*   *The Rise of AI in India: Startups, Sovereign Tech, and the Future of Work*

---

### **Target Keywords:**
*   *Primary:* Rise of AI in India, AI in India, Indian AI startups
*   *Secondary:* IndiaAI Mission, Indic LLMs, AI in Indian healthcare, tech talent in India, digital revolution India

---

## **Detailed Blog Outline**

### **1. Introduction (Approx. 150–200 words)**
*   **The Hook:** Start with a compelling statistic or real-world scenario (e.g., a farmer in rural India 

In [73]:
print(final_state['rating'])

10
